In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# Configuration
# ============================================================

BASE_DIR = Path(".")

MODELS = ["LSTM", "CNN", "LR"]

TICKERS = [
    "AAPL",
    "QCOM",
    "BNY",
    "C",
    "0097.KL",
    "0166.KL",
    "1023.KL",
    "5258.KL",
]

FEATURES = [
    "Close",
    "20MA",
    "50MA",
    "RSI",
    "MACD",
    "Lower_BB",
    "Upper_BB",
    "ATR",
    "CCI",
    "OBV",
    "Williams_%R",
]

OUTPUT_COLUMNS = [
    "Model",
    "Types",
    "Ticker",
] + FEATURES

OUTPUT_FILE = "all_models_shap_lime_feature_comparison.csv"


# ============================================================
# Normalize feature names
# ============================================================

def normalize_feature_name(feature):
    """
    Convert variations of feature names into the requested format.
    """

    if pd.isna(feature):
        return None

    original = str(feature).strip()

    key = (
        original.lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
    )

    mapping = {
        "close": "Close",

        "20ma": "20MA",
        "ma20": "20MA",
        "sma20": "20MA",

        "50ma": "50MA",
        "ma50": "50MA",
        "sma50": "50MA",

        "rsi": "RSI",
        "macd": "MACD",

        "lowerbb": "Lower_BB",
        "bblower": "Lower_BB",
        "lowerbollingerband": "Lower_BB",

        "upperbb": "Upper_BB",
        "bbupper": "Upper_BB",
        "upperbollingerband": "Upper_BB",

        "atr": "ATR",
        "cci": "CCI",
        "obv": "OBV",

        "williams%r": "Williams_%R",
        "williamsr": "Williams_%R",
        "willr": "Williams_%R",
        "williamspercentr": "Williams_%R",
    }

    return mapping.get(key, original)


# ============================================================
# Detect ticker from folder name
# ============================================================

def get_ticker_from_path(file_path):
    """
    Example:
        LSTM/AAPL_XAI/shap_lime_feature_comparison.csv
        LSTM/0097.KL_XAI/shap_lime_feature_comparison.csv

    returns:
        AAPL
        0097.KL
    """

    folder_name = file_path.parent.name
    #print(folder_name)
    # Remove _XAI / XAI ending
    ticker = folder_name

    if ticker.upper().endswith("_LSTM_XAI"):
        ticker = ticker[:-9]

    elif ticker.upper().endswith("_CNN_XAI"):
        ticker = ticker[:-8]

    elif ticker.upper().endswith("_LR_XAI"):
        ticker = ticker[:-7]

    ticker = ticker.rstrip("_-")
    #print(ticker)
    # Match against required tickers
    for valid_ticker in TICKERS:
        if ticker.upper() == valid_ticker.upper():
            return valid_ticker

    # Fallback: search ticker inside folder name
    #for valid_ticker in TICKERS:
    #    if valid_ticker.upper() in folder_name.upper():
    #        return valid_ticker

    return None


# ============================================================
# Find important source columns
# ============================================================

def find_column(df, possible_names):
    """
    Find a column regardless of spaces, underscores and case.
    """

    normalized_columns = {}

    for col in df.columns:
        key = (
            str(col)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        normalized_columns[key] = col

    for name in possible_names:
        key = (
            name.lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        if key in normalized_columns:
            return normalized_columns[key]

    return None


# ============================================================
# Read all files
# ============================================================

rows = []

for model in MODELS:

    files = list(
        (BASE_DIR / model).glob(
            "*XAI/shap_lime_feature_comparison.csv"
        )
    )

    print(f"\n{model}: {len(files)} file(s) found")

    for file_path in files:

        print(f"Reading: {file_path}")

        ticker = get_ticker_from_path(file_path)

        if ticker is None:
            print(
                f"  WARNING: Cannot determine ticker from "
                f"{file_path.parent.name}. Skipping."
            )
            continue

        try:
            df = pd.read_csv(file_path)

        except Exception as e:
            print(f"  ERROR: {e}")
            continue

        # ----------------------------------------------------
        # Find source columns
        # ----------------------------------------------------

        feature_col = find_column(
            df,
            [
                "Feature",
                "Features",
                "Feature Name",
            ]
        )

        shap_col = find_column(
            df,
            [
                "SHAP Importance",
                "SHAP Importance Value",
                "SHAP Mean Abs",
                "SHAP MeanAbs",
                "SHAP Importance (common samples)",
            ]
        )

        lime_col = find_column(
            df,
            [
                "LIME Importance",
                "LIME Importance",
                "LIME Mean Abs",
                "LIME MeanAbs",
                "LIME Contribution Importance",
            ]
        )

        if feature_col is None:
            print("  WARNING: Feature column not found.")
            continue

        if shap_col is None:
            print("  WARNING: SHAP Importance column not found.")

        if lime_col is None:
            print("  WARNING: LIME Importance column not found.")

        # Normalize feature names
        df["_Feature"] = df[feature_col].apply(
            normalize_feature_name
        )

        # ====================================================
        # SHAP row
        # ====================================================

        shap_row = {
            "Model": model,
            "Types": "SHAP",
            "Ticker": ticker,
        }

        for feature in FEATURES:

            if shap_col is None:
                shap_row[feature] = np.nan
                continue

            matched = df.loc[
                df["_Feature"] == feature,
                shap_col
            ]

            if not matched.empty:
                shap_row[feature] = matched.iloc[0]
            else:
                shap_row[feature] = np.nan

        rows.append(shap_row)

        # ====================================================
        # LIME row
        # ====================================================

        lime_row = {
            "Model": model,
            "Types": "LIME",
            "Ticker": ticker,
        }

        for feature in FEATURES:

            if lime_col is None:
                lime_row[feature] = np.nan
                continue

            matched = df.loc[
                df["_Feature"] == feature,
                lime_col
            ]

            if not matched.empty:
                lime_row[feature] = matched.iloc[0]
            else:
                lime_row[feature] = np.nan

        rows.append(lime_row)


# ============================================================
# Create dataframe
# ============================================================

result = pd.DataFrame(
    rows,
    columns=OUTPUT_COLUMNS
)


# ============================================================
# Ensure ALL combinations exist
#
# 3 models × 2 types × 8 tickers = 48 rows
# ============================================================

expected = pd.MultiIndex.from_product(
    [
        MODELS,
        ["SHAP", "LIME"],
        TICKERS,
    ],
    names=[
        "Model",
        "Types",
        "Ticker",
    ]
).to_frame(index=False)


result = expected.merge(
    result,
    on=[
        "Model",
        "Types",
        "Ticker",
    ],
    how="left"
)


# ============================================================
# Sort
# ============================================================

result["Model"] = pd.Categorical(
    result["Model"],
    categories=MODELS,
    ordered=True
)

result["Types"] = pd.Categorical(
    result["Types"],
    categories=["SHAP", "LIME"],
    ordered=True
)

result["Ticker"] = pd.Categorical(
    result["Ticker"],
    categories=TICKERS,
    ordered=True
)


result = (
    result
    .sort_values(
        ["Model", "Types", "Ticker"]
    )
    .reset_index(drop=True)
)


# ============================================================
# Restore strings
# ============================================================

result["Model"] = result["Model"].astype(str)
result["Types"] = result["Types"].astype(str)
result["Ticker"] = result["Ticker"].astype(str)


# ============================================================
# Final column order
# ============================================================

result = result[OUTPUT_COLUMNS]


# ============================================================
# Save CSV
# ============================================================

result.to_csv(
    OUTPUT_FILE,
    index=False,
    na_rep="na"
)


print("\n========================================")
print("Completed")
print("========================================")

print(result.to_string(index=False))

print(f"\nSaved to: {OUTPUT_FILE}")
print(f"Total rows: {len(result)}")


LSTM: 8 file(s) found
Reading: LSTM\0097.KL_LSTM_XAI\shap_lime_feature_comparison.csv
Reading: LSTM\0166.KL_LSTM_XAI\shap_lime_feature_comparison.csv
Reading: LSTM\1023.KL_LSTM_XAI\shap_lime_feature_comparison.csv
Reading: LSTM\5258.KL_LSTM_XAI\shap_lime_feature_comparison.csv
Reading: LSTM\AAPL_LSTM_XAI\shap_lime_feature_comparison.csv
Reading: LSTM\BNY_LSTM_XAI\shap_lime_feature_comparison.csv
Reading: LSTM\C_LSTM_XAI\shap_lime_feature_comparison.csv
Reading: LSTM\QCOM_LSTM_XAI\shap_lime_feature_comparison.csv

CNN: 8 file(s) found
Reading: CNN\0097.KL_CNN_XAI\shap_lime_feature_comparison.csv
Reading: CNN\0166.KL_CNN_XAI\shap_lime_feature_comparison.csv
Reading: CNN\1023.KL_CNN_XAI\shap_lime_feature_comparison.csv
Reading: CNN\5258.KL_CNN_XAI\shap_lime_feature_comparison.csv
Reading: CNN\AAPL_CNN_XAI\shap_lime_feature_comparison.csv
Reading: CNN\BNY_CNN_XAI\shap_lime_feature_comparison.csv
Reading: CNN\C_CNN_XAI\shap_lime_feature_comparison.csv
Reading: CNN\QCOM_CNN_XAI\shap_lime_fe